# Comparing runs with 16 and 32 frames on WLASL 100 to 2000, with similar parameters

In [11]:
import json
from pathlib import Path
from typing import cast

#locals
# from code.run_types import ResSet, RunRes
import pandas as pd

from src.resulting import RESULTS_DIR, load_config_and_find_runs
from src.results.satnac_2026.filters import additional_modifications, exclude_keys
from src.run_types import GenInfo


def load_find(conf_path: Path) -> GenInfo:
    runs =  load_config_and_find_runs(
            conf_path,
            exclude=exclude_keys,
            extra_mods=additional_modifications,
        )
    assert runs is not None
    return runs

In [12]:
results_dir = RESULTS_DIR / 'satnac_2026'

target_lengths = [16]

runs_paths = {
    tl : results_dir / f'spec_{tl}f_50p.toml'
    for tl in target_lengths
}

for runs_p in runs_paths.values():
    assert runs_p.exists(), f"{runs_p} not found"


## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [13]:
runs_by_tl = {
    tl: load_find(runs_p) 
    for tl, runs_p in runs_paths.items()
}

INFO resulting: Loaded que state from /home/luke/Code/SLR/src/que/Runs.json
INFO resulting: Found 33/311 runs matching the spec
INFO resulting: Excluded 2 runs based on additional modifications


## Runs with different number of frames:

Lets check how many runs there are that match that spec:

In [14]:
for tl, runs in runs_by_tl.items():
    print(f'Target length: {tl}')
    print(f"Runs keys: {runs.keys()}")
    print(f"Number of entries: {len(runs['results'])}\n")

# print(json.dumps(runs_16['results'][0], indent=4))


Target length: 16
Runs keys: dict_keys(['spec', 'results'])
Number of entries: 31



## Now we can compare the runs

In [15]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

#### Load into data frame, including info such as number of frames

In [16]:
df_format = []

for tl, runs in runs_by_tl.items():
    for res in runs["results"]:  
        model_name = res["admin"]["model"]
        df_format.append(
            {
                "model": model_name,
                "exp no": res['admin']['exp_no'],
                "run_id": res['wandb']['run_id'],
                "subset": res["admin"]["split"],
                "No. frames": tl
            }
            
            | {k: v for k, v in res["results"][set_name][acc_type].items()}
            | {"config path": res["admin"]["config_path"],
               "weight path": res['admin']['weight_path']}
        )    

df = pd.DataFrame(df_format) 

In [17]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
# df

In [18]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [19]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']
for set_name in subsets:
    print(f'{set_name}'.capitalize())
    subdf = df[df['subset'] == set_name]
    display(subdf.sort_values('Top-1', ascending=False))

Asl100


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
30,MViTv2_S,007,None,asl100,16,79.46,91.86,95.35,configfiles/generic/lframe_hwd_warmrestarts.toml,None
23,MViTv1_B,003,j8v7g110,asl100,16,71.32,89.53,91.86,configfiles/asl100/MViTv1_B/exp003.toml,None
25,Swin3D_S,003,0znkyo6w,asl100,16,66.67,87.21,91.86,configfiles/asl100/Swin3D_S/exp003.toml,None
26,Swin3D_T,003,kqbvbgsr,asl100,16,60.08,85.27,93.80,configfiles/asl100/Swin3D_T/exp003.toml,None
29,R3D_18,006,56o6x69m,asl100,16,55.04,82.17,88.76,configfiles/asl100/R3D_18/exp006.toml,None
24,Swin3D_B,003,wo8fe3st,asl100,16,54.26,82.56,91.09,configfiles/asl100/Swin3D_B/exp003.toml,None
1,S3D,051,f6r2dos3,asl100,16,53.10,79.84,87.98,configfiles/asl100/S3D/exp046.toml,None
27,S3D,046,5xgru0cs,asl100,16,50.39,77.91,86.43,configfiles/asl100/S3D/exp046.toml,None
28,R(2+1)D_18,007,v33pxh3s,asl100,16,48.06,77.13,86.05,configfiles/asl100/R(2+1)D_18/exp007.toml,None


Asl300


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
18,MViTv2_S,001,nfwehytd,asl300,16,65.42,88.92,93.56,configfiles/asl300/MViTv2_S/exp001.toml,None
19,MViTv1_B,001,8noyy5tz,asl300,16,58.98,84.88,89.82,configfiles/asl300/MViTv1_B/exp001.toml,None
15,Swin3D_T,002,qx2nc2mj,asl300,16,53.74,81.59,89.07,configfiles/asl300/Swin3D_T/exp002.toml,None
22,S3D,002,vty80mdi,asl300,16,51.95,78.44,87.13,configfiles/asl300/S3D/exp002.toml,None
16,Swin3D_S,002,hqxu4w1r,asl300,16,50.60,80.39,87.57,configfiles/asl300/Swin3D_S/exp002.toml,None
17,Swin3D_B,002,bjc1br16,asl300,16,50.30,78.14,87.87,configfiles/asl300/Swin3D_B/exp002.toml,None
20,R(2+1)D_18,002,7vrz6dvj,asl300,16,38.02,67.07,77.10,configfiles/asl300/R(2+1)D_18/exp002.toml,None
21,R3D_18,002,xmadiwzp,asl300,16,25.60,54.64,63.47,configfiles/asl300/R3D_18/exp002.toml,None


Asl1000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
10,Swin3D_S,000,n27qc5m6,asl1000,16,8.80,25.16,37.47,configfiles/asl1000/Swin3D_S/exp000.toml,None
12,MViTv2_S,000,rc5m3meh,asl1000,16,53.41,81.77,87.79,configfiles/asl1000/MViTv2_S/exp000.toml,None
13,MViTv1_B,000,cdua0ddj,asl1000,16,46.06,76.81,83.58,configfiles/asl1000/MViTv1_B/exp000.toml,None
14,S3D,000,kpdu2hra,asl1000,16,36.25,67.80,77.08,configfiles/asl1000/S3D/exp000.toml,None
11,Swin3D_B,000,94kuihol,asl1000,16,33.74,65.30,74.41,configfiles/asl1000/Swin3D_B/exp000.toml,None
9,Swin3D_T,000,rdmmkptp,asl1000,16,23.24,53.46,65.78,configfiles/asl1000/Swin3D_T/exp000.toml,None


Asl2000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
6,R(2+1)D_18,000,mi1aom4x,asl2000,16,6.32,18.20,25.63,configfiles/asl2000/R(2+1)D_18/exp000.toml,None
3,R3D_18,000,sz27ivxv,asl2000,16,6.15,17.65,25.63,configfiles/asl2000/R3D_18/exp000.toml,None
4,Swin3D_B,000,x3n0l0p0,asl2000,16,4.48,13.75,21.22,configfiles/asl2000/Swin3D_B/exp000.toml,None
7,MViTv2_S,000,fkv6kpik,asl2000,16,38.87,71.62,79.82,configfiles/asl2000/MViTv2_S/exp000.toml,None
0,Swin3D_T,000,kct76mvs,asl2000,16,3.09,11.15,17.65,configfiles/asl2000/Swin3D_T/exp000.toml,None
2,Swin3D_S,000,cxh3q986,asl2000,16,20.74,46.37,59.19,configfiles/asl2000/Swin3D_S/exp000.toml,None
8,MViTv1_B,000,b73or9ez,asl2000,16,19.94,46.96,59.64,configfiles/asl2000/MViTv1_B/exp000.toml,None
5,S3D,000,olp97b32,asl2000,16,19.62,46.96,58.25,configfiles/asl2000/S3D/exp000.toml,None


In [20]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']

# unique (model, frame count) combos, in a stable order
combos = df[['model', 'No. frames']].drop_duplicates().sort_values(['model', 'No. frames'])

for _, combo in combos.iterrows():
    model, frames = combo['model'], combo['No. frames']
    model_escaped = model.replace('_', '\\_')
    print(f'{model_escaped} ({frames} frames)')

    for set_name in subsets:
        subdf = df[(df['model'] == model) &
                   (df['No. frames'] == frames) &
                   (df['subset'] == set_name)]
        if subdf.empty:
            print('        & - & - & -')
            continue
        best = subdf.sort_values('Top-1', ascending=False).iloc[0]
        print(f"        & {best['Top-1']} & {best['Top-5']} & {best['Top-10']}")
    print()

MViTv1\_B (16 frames)
        & 71.32 & 89.53 & 91.86
        & 58.98 & 84.88 & 89.82
        & 46.06 & 76.81 & 83.58
        & 19.94 & 46.96 & 59.64

MViTv2\_S (16 frames)
        & 79.46 & 91.86 & 95.35
        & 65.42 & 88.92 & 93.56
        & 53.41 & 81.77 & 87.79
        & 38.87 & 71.62 & 79.82

R(2+1)D\_18 (16 frames)
        & 48.06 & 77.13 & 86.05
        & 38.02 & 67.07 & 77.10
        & - & - & -
        & 6.32 & 18.20 & 25.63

R3D\_18 (16 frames)
        & 55.04 & 82.17 & 88.76
        & 25.60 & 54.64 & 63.47
        & - & - & -
        & 6.15 & 17.65 & 25.63

S3D (16 frames)
        & 53.10 & 79.84 & 87.98
        & 51.95 & 78.44 & 87.13
        & 36.25 & 67.80 & 77.08
        & 19.62 & 46.96 & 58.25

Swin3D\_B (16 frames)
        & 54.26 & 82.56 & 91.09
        & 50.30 & 78.14 & 87.87
        & 33.74 & 65.30 & 74.41
        & 4.48 & 13.75 & 21.22

Swin3D\_S (16 frames)
        & 66.67 & 87.21 & 91.86
        & 50.60 & 80.39 & 87.57
        & 8.80 & 25.16 & 37.47
        & 